In [3]:
import torch
import random
import numpy as np
import pandas as pd 
from torch.utils.data import Dataset, DataLoader

In [2]:
#physio_path = '/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_data_standardized_nonan.csv'
physio_path = '/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/data_with_cfc.csv'
physio_data = pd.read_csv(physio_path, index_col=False)
physio_data = physio_data.iloc[:, 1:]
physio_data

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet_x,Diet_y,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-40-2162,28,1611,-0.299075,-0.298795,-0.299221,-0.281284,-0.285116,-0.292757,-0.238758,...,-0.279397,-0.302223,-0.302210,-0.302207,1,40,40,DO-40-2162,3.78,0
1,DO-40-2179,28,1557,-0.298358,-0.299227,-0.299235,-0.281751,-0.293085,-0.293425,-0.239182,...,-0.275674,-0.302204,-0.302197,-0.302197,1,40,40,DO-40-2179,26.71,2
2,DO-40-2181,28,1533,-0.299589,-0.299802,-0.299617,-0.281751,-0.290396,-0.294525,-0.235990,...,-0.282036,-0.302218,-0.302197,-0.302202,1,40,40,DO-40-2181,27.02,2
3,DO-40-2187,28,1520,-0.298924,-0.298507,-0.298382,-0.285984,-0.291374,-0.293181,-0.242950,...,-0.275674,-0.302213,-0.302205,-0.302199,1,40,40,DO-40-2187,50.40,5
4,DO-40-2099,26,1638,-0.299145,-0.299223,-0.299314,-0.281054,-0.292984,-0.294603,-0.243130,...,-0.282728,-0.302213,-0.302207,-0.302191,1,40,40,DO-40-2099,34.18,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,DO-2D-4007,22,771,-0.297123,-0.296254,-0.293633,-0.287916,-0.281208,-0.268981,-0.231718,...,-0.294808,-0.302189,-0.302200,-0.302156,3,2D,2D,DO-2D-4007,23.69,2
500,DO-1D-3030,22,748,-0.299135,-0.298423,-0.293633,-0.284512,-0.284879,-0.268981,-0.233019,...,-0.294808,-0.302189,-0.302216,-0.302156,4,1D,1D,DO-1D-3030,29.78,2
501,DO-20-1021,22,738,-0.298310,-0.297380,-0.293633,-0.280750,-0.282837,-0.268981,-0.233242,...,-0.294808,-0.302189,-0.302198,-0.302156,2,20,20,DO-20-1021,18.22,1
502,DO-AL-0002,22,755,-0.298379,-0.296017,-0.293633,-0.284551,-0.286319,-0.268981,-0.241560,...,-0.294808,-0.302189,-0.302192,-0.302156,5,AL,AL,DO-AL-0002,57.42,5


In [27]:
np.min(physio_data.iloc[:, 1:-2]), np.max(physio_data.iloc[:, 3:-2])

(-0.3128270086048231, 7.107052564259018)

In [28]:
physio_var = physio_data.iloc[:, 1:-1].var(axis=0)
physio_var

Generation                4.644642e+00
SurvDays                  7.912356e+04
Y1A_BW_BW                 7.714110e-07
Y2A_BW_BW                 4.055867e-06
Y3A_BW_BW                 1.050198e-05
Y1_Glu.F_Glucose          3.907626e-05
Y2_Glu.F_Glucose          8.940893e-05
Y3_Glu.F_Glucose          1.783399e-04
Y1_Echo_BPM               1.126723e-04
Y2_Echo_BPM               1.575102e-04
Y3_Echo_BPM               1.259674e-04
Y1_Echo_CardiacOutput     1.251889e+00
Y2_Echo_CardiacOutput     2.253209e+00
Y3_Echo_CardiacOutput     3.134395e+00
Y1_CBC_Hgb                1.824325e-07
Y2_CBC_Hgb                7.084397e-07
Y3_CBC_Hgb                7.593765e-07
Y1_Rotarod_Mean           1.768308e-04
Y2_Rotarod_Mean           2.448031e-04
Y3_Rotarod_Mean           1.639053e-04
Y1_AS_MeanLog             1.928002e-09
Y2_AS_MeanLog             3.660646e-09
Y3_AS_MeanLog             8.982483e-10
Y1_Wheel_AvgSpeedLFC      5.653428e-08
Y2_Wheel_AvgSpeedLFC      2.549442e-08
Y3_Wheel_AvgSpeedLFC     

In [31]:
pd.DataFrame(physio_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/physio_var.csv', index=False)

In [29]:
len(physio_var)

36

In [4]:
genoprobs_path = '/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/genoprobs_proj.csv'
genoprobs = pd.read_csv(genoprobs_path)
genoprobs

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-9.608385,1.688261,2.132044,6.027388,-1.819352,-14.918910,0.255752,-12.551320,-32.210761,-18.746747
1,DO-1D-3002,-2.628764,22.933458,7.645753,5.538203,7.347764,2.942720,11.347963,4.449972,-6.118222,...,7.449434,0.517254,19.643941,-1.064277,7.085058,-14.735193,3.369756,-20.902948,11.984909,-0.489731
2,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,3.795336,8.341982,7.015957,-4.252234,17.039490,-25.052406,-2.387492,27.084163,16.082832,-7.912733
3,DO-1D-3004,-6.680621,-8.019135,3.125651,4.316594,11.230175,25.391169,2.784092,5.191860,0.504001,...,6.408988,-1.998942,5.332627,11.780933,0.869926,-2.390750,-14.677933,-7.229061,1.885609,-14.551324
4,DO-1D-3005,-7.762874,6.568555,24.752748,7.915682,8.565950,1.527536,-6.604914,12.055185,-5.745179,...,11.553945,4.265048,-6.110699,-3.763212,7.669412,-15.097044,-26.173256,31.715402,-14.482437,19.841158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-2.603553,3.989595,5.078461,-12.081190,17.994081,1.477907,8.555104,15.169529,-11.607028,...,9.336485,-6.130255,1.278046,-5.116948,3.087162,-5.732800,0.773564,-15.927428,-3.684303,-8.617224
942,DO-AL-0064,-16.592022,-13.050886,-2.775860,6.993123,22.905117,-23.698464,-6.764692,3.005287,-21.190046,...,-24.039623,12.675260,40.292406,-15.214119,5.056168,15.422292,-13.980632,-3.702247,4.333789,6.259631
943,DO-AL-0073,37.893878,-8.674703,9.040660,12.313616,-18.294495,6.359014,-2.191113,-2.120036,-3.447605,...,29.586744,19.535960,18.001992,14.377207,9.018423,20.694376,24.442459,6.656219,-3.966847,7.416123
944,DO-AL-0089,-9.300763,-7.110903,-9.108345,0.429022,26.645567,-21.359485,10.953367,22.687536,10.444933,...,-13.349453,-0.512996,1.010019,7.261418,-33.304929,7.421048,18.690578,10.997129,2.577963,-22.869207


In [20]:
np.min(genoprobs.iloc[:, 1:]), np.max(genoprobs.iloc[:, 1:])

(-73.2288948863759, 64.73026584617132)

In [33]:
genoprobs_var = genoprobs.iloc[:, 1:].var(axis=0)
genoprobs_var

0       159.708584
1       142.847381
2       143.785389
3       165.711994
4       151.712388
           ...    
5868    170.743224
5869    143.170025
5870    160.085848
5871    142.560699
5872    149.217425
Length: 5873, dtype: float64

In [34]:
pd.DataFrame(genoprobs_var.values).to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/genoprobs_var.csv', index=False)

In [6]:
genoprobs = genoprobs.rename(columns={genoprobs.columns[0]: 'MouseID'})
genoprobs

,MouseID,0,1,2,3,4,5,6,7,8,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-9.608385,1.688261,2.132044,6.027388,-1.819352,-14.918910,0.255752,-12.551320,-32.210761,-18.746747
1,DO-1D-3002,-2.628764,22.933458,7.645753,5.538203,7.347764,2.942720,11.347963,4.449972,-6.118222,...,7.449434,0.517254,19.643941,-1.064277,7.085058,-14.735193,3.369756,-20.902948,11.984909,-0.489731
2,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,3.795336,8.341982,7.015957,-4.252234,17.039490,-25.052406,-2.387492,27.084163,16.082832,-7.912733
3,DO-1D-3004,-6.680621,-8.019135,3.125651,4.316594,11.230175,25.391169,2.784092,5.191860,0.504001,...,6.408988,-1.998942,5.332627,11.780933,0.869926,-2.390750,-14.677933,-7.229061,1.885609,-14.551324
4,DO-1D-3005,-7.762874,6.568555,24.752748,7.915682,8.565950,1.527536,-6.604914,12.055185,-5.745179,...,11.553945,4.265048,-6.110699,-3.763212,7.669412,-15.097044,-26.173256,31.715402,-14.482437,19.841158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,DO-AL-0035,-2.603553,3.989595,5.078461,-12.081190,17.994081,1.477907,8.555104,15.169529,-11.607028,...,9.336485,-6.130255,1.278046,-5.116948,3.087162,-5.732800,0.773564,-15.927428,-3.684303,-8.617224
942,DO-AL-0064,-16.592022,-13.050886,-2.775860,6.993123,22.905117,-23.698464,-6.764692,3.005287,-21.190046,...,-24.039623,12.675260,40.292406,-15.214119,5.056168,15.422292,-13.980632,-3.702247,4.333789,6.259631
943,DO-AL-0073,37.893878,-8.674703,9.040660,12.313616,-18.294495,6.359014,-2.191113,-2.120036,-3.447605,...,29.586744,19.535960,18.001992,14.377207,9.018423,20.694376,24.442459,6.656219,-3.966847,7.416123
944,DO-AL-0089,-9.300763,-7.110903,-9.108345,0.429022,26.645567,-21.359485,10.953367,22.687536,10.444933,...,-13.349453,-0.512996,1.010019,7.261418,-33.304929,7.421048,18.690578,10.997129,2.577963,-22.869207


In [5]:
physio_data['MouseID'].isin(genoprobs['MouseID']).sum()

np.int64(929)

In [6]:
physio_data['MouseID'].dtypes

dtype('O')

In [7]:
# physio_data['MouseID'] = physio_data['MouseID'].astype(str)
# genoprobs['MouseID'] = genoprobs['MouseID'].astype(str)

In [8]:
genoprobs['MouseID'].dtypes

dtype('O')

In [9]:
physio_data['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [10]:
genoprobs['MouseID'].map(type).unique()

array([<class 'str'>], dtype=object)

In [7]:
joined_genetic_physio = pd.merge(genoprobs, physio_data, on='MouseID', how='inner')
joined_genetic_physio

,MouseID,0,1,2,3,4,5,6,7,8,...,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet_x,Diet_y,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-0.284390,-0.302189,-0.302202,-0.302199,4,1D,1D,DO-1D-3001,0.84,0
1,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,-0.287381,-0.302189,-0.302203,-0.302198,4,1D,1D,DO-1D-3003,83.69,8
2,DO-1D-3010,1.859422,-7.514837,1.146809,-10.576791,-14.123598,0.187016,2.846669,-21.453514,-14.552152,...,-0.284049,-0.302189,-0.302203,-0.302205,4,1D,1D,DO-1D-3010,36.71,3
3,DO-1D-3011,6.336620,-19.787761,9.017313,-4.479619,12.846408,-5.231286,16.983432,-16.252265,-12.688484,...,-0.281171,-0.302189,-0.302205,-0.302193,4,1D,1D,DO-1D-3011,21.96,2
4,DO-1D-3013,-23.926661,-7.046359,8.830735,1.312327,-0.527705,6.687224,-9.731708,6.568401,6.597907,...,-0.294808,-0.302189,-0.302202,-0.302156,4,1D,1D,DO-1D-3013,6.93,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,DO-AL-0186,-5.162836,8.648018,-1.372207,10.970472,-1.921667,-3.733688,8.676810,-16.461100,-8.221440,...,-0.294808,-0.302211,-0.302210,-0.302156,5,AL,AL,DO-AL-0186,20.44,2
497,DO-AL-0188,11.180352,-8.517944,8.474807,-6.834697,12.355449,20.439765,11.830693,-11.656498,-24.270286,...,-0.283513,-0.302216,-0.302200,-0.302202,5,AL,AL,DO-AL-0188,18.31,1
498,DO-AL-0191,3.404846,0.426192,-2.073981,-12.920995,-19.462683,6.055004,9.646308,4.292594,10.025039,...,-0.294808,-0.302207,-0.302194,-0.302156,5,AL,AL,DO-AL-0191,24.71,2
499,DO-1D-3083,-20.852529,7.764088,-5.252014,8.705888,-11.405415,13.622378,1.249871,8.666080,14.656721,...,-0.277799,-0.302223,-0.302210,-0.302199,4,1D,1D,DO-1D-3083,31.07,3


In [13]:
joined_genetic_physio.iloc[:, 1:5874]

,0,1,2,3,4,5,6,7,8,9,...,5863,5864,5865,5866,5867,5868,5869,5870,5871,5872
0,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,16.481526,...,-9.608385,1.688261,2.132044,6.027388,-1.819352,-14.918910,0.255752,-12.551320,-32.210761,-18.746747
1,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,-14.451874,...,3.795336,8.341982,7.015957,-4.252234,17.039490,-25.052406,-2.387492,27.084163,16.082832,-7.912733
2,1.859422,-7.514837,1.146809,-10.576791,-14.123598,0.187016,2.846669,-21.453514,-14.552152,-3.654468,...,9.519715,-7.268057,3.741343,-9.417331,-1.049166,-5.498758,-5.367425,9.114800,3.424338,-0.379229
3,6.336620,-19.787761,9.017313,-4.479619,12.846408,-5.231286,16.983432,-16.252265,-12.688484,8.676695,...,13.925967,-0.000577,-32.550632,13.622736,-6.793734,-7.893204,8.950426,8.592419,-7.557427,-10.798918
4,-23.926661,-7.046359,8.830735,1.312327,-0.527705,6.687224,-9.731708,6.568401,6.597907,-26.565523,...,12.601419,-17.014651,4.202860,-31.467534,-0.954875,8.622592,-14.107947,15.045199,-0.035120,-6.854894
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,-5.162836,8.648018,-1.372207,10.970472,-1.921667,-3.733688,8.676810,-16.461100,-8.221440,-0.572300,...,7.600915,1.824296,5.668207,-19.346480,4.554441,-10.431488,-11.383410,12.908321,5.243945,4.119316
497,11.180352,-8.517944,8.474807,-6.834697,12.355449,20.439765,11.830693,-11.656498,-24.270286,4.732213,...,-10.008184,6.568750,-7.533109,10.252428,0.500679,-19.298980,-10.240820,-10.535206,-4.789270,6.239094
498,3.404846,0.426192,-2.073981,-12.920995,-19.462683,6.055004,9.646308,4.292594,10.025039,-9.331366,...,-11.741685,14.823899,-14.705902,-3.329089,3.390845,-10.066433,-1.425435,-17.961321,2.114508,9.352855
499,-20.852529,7.764088,-5.252014,8.705888,-11.405415,13.622378,1.249871,8.666080,14.656721,-19.723504,...,-8.721728,-26.027744,-26.663276,2.131347,-15.731743,-1.064788,-10.416841,-12.688912,2.477686,5.573192


In [17]:
joined_genetic_physio.iloc[:, 5874:-6]

,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,Y2_Echo_BPM,...,Y3_Wheel_AvgSpeedLFC,Y1_Wheel_AvgDistLFC,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj
0,22,895,-0.298444,-0.297323,-0.297680,-0.286205,-0.287075,-0.287697,-0.237677,-0.248171,...,-0.302421,-0.302246,-0.302088,-0.302575,-0.287930,-0.282162,-0.284390,-0.302189,-0.302202,-0.302199
1,22,925,-0.299745,-0.299222,-0.298910,-0.281316,-0.282797,-0.280730,-0.234229,-0.254524,...,-0.302421,-0.302086,-0.301963,-0.302575,-0.282807,-0.284586,-0.287381,-0.302189,-0.302203,-0.302198
2,22,1166,-0.300000,-0.299245,-0.298769,-0.283760,-0.284508,-0.285619,-0.237237,-0.244407,...,-0.302186,-0.301843,-0.301781,-0.301919,-0.280937,-0.283942,-0.284049,-0.302189,-0.302203,-0.302205
3,22,1042,-0.298834,-0.297755,-0.297675,-0.282294,-0.284875,-0.279019,-0.232997,-0.231096,...,-0.302421,-0.301844,-0.301921,-0.302575,-0.284704,-0.281745,-0.281171,-0.302189,-0.302205,-0.302193
4,22,811,-0.298503,-0.297819,-0.298579,-0.283760,-0.286586,-0.268981,-0.240292,-0.243785,...,-0.302421,-0.302190,-0.302036,-0.302575,-0.286629,-0.283412,-0.294808,-0.302189,-0.302202,-0.302156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,28,735,-0.298718,-0.297217,-0.293633,-0.287006,-0.280252,-0.268981,-0.233249,-0.235801,...,-0.302421,-0.302020,-0.302180,-0.302575,-0.279690,-0.281315,-0.294808,-0.302211,-0.302210,-0.302156
497,28,973,-0.299543,-0.297733,-0.297337,-0.282362,-0.285141,-0.288658,-0.238521,-0.234555,...,-0.302421,-0.302083,-0.302108,-0.302575,-0.284575,-0.280596,-0.283513,-0.302216,-0.302200,-0.302202
498,28,832,-0.298737,-0.296975,-0.296712,-0.283584,-0.283552,-0.268981,-0.243609,-0.241071,...,-0.302421,-0.302059,-0.302246,-0.302575,-0.273897,-0.282754,-0.294808,-0.302207,-0.302194,-0.302156
499,24,1017,-0.298324,-0.297026,-0.296605,-0.280185,-0.269761,-0.279112,-0.238602,-0.241634,...,-0.302421,-0.302188,-0.301821,-0.302575,-0.271600,-0.274878,-0.277799,-0.302223,-0.302210,-0.302199


In [12]:
#joined_genetic_physio.to_csv('/nfs/turbo/umms-adraelos/caloric_restriction_DO_mice/Genotype_data/joined_genetic_physio.csv', index=False)

In [19]:
joined_genetic_physio = joined_genetic_physio.drop('Diet_y', axis=1)
joined_genetic_physio = joined_genetic_physio.rename(columns={'Diet_x': 'Diet'})

In [20]:
joined_genetic_physio

,MouseID,0,1,2,3,4,5,6,7,8,...,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-1D-3001,11.181311,12.383703,13.223754,18.683861,-3.105873,15.071812,4.566722,-8.317668,15.178740,...,-0.282162,-0.284390,-0.302189,-0.302202,-0.302199,4,1D,DO-1D-3001,0.84,0
1,DO-1D-3003,14.611459,-0.662600,-24.308638,-22.627848,-18.261186,1.859805,6.543754,22.646550,-1.809972,...,-0.284586,-0.287381,-0.302189,-0.302203,-0.302198,4,1D,DO-1D-3003,83.69,8
2,DO-1D-3010,1.859422,-7.514837,1.146809,-10.576791,-14.123598,0.187016,2.846669,-21.453514,-14.552152,...,-0.283942,-0.284049,-0.302189,-0.302203,-0.302205,4,1D,DO-1D-3010,36.71,3
3,DO-1D-3011,6.336620,-19.787761,9.017313,-4.479619,12.846408,-5.231286,16.983432,-16.252265,-12.688484,...,-0.281745,-0.281171,-0.302189,-0.302205,-0.302193,4,1D,DO-1D-3011,21.96,2
4,DO-1D-3013,-23.926661,-7.046359,8.830735,1.312327,-0.527705,6.687224,-9.731708,6.568401,6.597907,...,-0.283412,-0.294808,-0.302189,-0.302202,-0.302156,4,1D,DO-1D-3013,6.93,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,DO-AL-0186,-5.162836,8.648018,-1.372207,10.970472,-1.921667,-3.733688,8.676810,-16.461100,-8.221440,...,-0.281315,-0.294808,-0.302211,-0.302210,-0.302156,5,AL,DO-AL-0186,20.44,2
497,DO-AL-0188,11.180352,-8.517944,8.474807,-6.834697,12.355449,20.439765,11.830693,-11.656498,-24.270286,...,-0.280596,-0.283513,-0.302216,-0.302200,-0.302202,5,AL,DO-AL-0188,18.31,1
498,DO-AL-0191,3.404846,0.426192,-2.073981,-12.920995,-19.462683,6.055004,9.646308,4.292594,10.025039,...,-0.282754,-0.294808,-0.302207,-0.302194,-0.302156,5,AL,DO-AL-0191,24.71,2
499,DO-1D-3083,-20.852529,7.764088,-5.252014,8.705888,-11.405415,13.622378,1.249871,8.666080,14.656721,...,-0.274878,-0.277799,-0.302223,-0.302210,-0.302199,4,1D,DO-1D-3083,31.07,3


In [21]:
joined_genetic_physio.to_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_cfc.csv', index=False)

In [6]:
joined_genetic_physio = pd.read_csv('/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/joined_genetic_physio_cfc.csv')

In [13]:
joined_genetic_physio.iloc[:, 1:genetic_dims]

NameError: name 'genetic_dims' is not defined

In [9]:
len(joined_genetic_physio.iloc[:, 5874:-4].columns)

36

In [ ]:
class PairedDODataset(Dataset):
    def __init__(self, paired_df, genetic_dims, diet_col, id_col, label_col): 
        self.paired_df = paired_df
        self.genetic_dims = genetic_dims
        self.diet_col = diet_col 
        self.id_col = id_col
        self.label_col = label_col

    def __len__(self):
        return len(self.paired_df)

    def __getitem__(self, idx):
        sample = self.paired_df.iloc[idx]

        # for the genoprobs data 
        genetic_df = torch.tensor(sample[1:self.genetic_dims].values.astype('float32'))
        genetic_label = sample[self.label_col]
        genetic_diet = sample[self.diet_col]
        genetic_id = sample[self.id_col]

        # for the physiological data 
        physio_df = torch.tensor(sample[self.genetic_dims:-2].values.astype('float32'))
        physio_label = sample[self.label_col]
        physio_diet = sample[self.diet_col]
        physio_id = sample[self.id_col]

        return (genetic_df, genetic_label, genetic_diet, genetic_id), (physio_df, physio_label, physio_diet, physio_id)

In [ ]:
genetic_dims = 5874
diet_col = 'Diet'
id_col = 'MouseID'
label_col = 'Diet_num'
dataset = PairedDODataset(joined_genetic_physio, genetic_dims, diet_col, id_col, label_col)

In [ ]:
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)

In [ ]:
first_iter = next(iter(dataloader))

In [ ]:
first_iter[0][0]

In [ ]:
first_iter[1][0].shape

In [ ]:
len(first_iter[1][3])